# SQL vs Excel: INN count check (with Impala connection)

Тетрадка:
- инициализирует подключение к Impala,
- считает в SQL за месяц:
  - `sql_agr_cnt` (уникальные договоры),
  - `sql_inn_cnt` (уникальные ИНН)
- результат можно сравнить с Excel (`rows`, `unique_inn`).

In [ ]:
import sys
from pathlib import Path
from rail_connectors.connection import connect

In [ ]:
# Подключаем путь к локальным секретам
ROOT = Path("E:/DTB(dashbord)")
SECRETS_DIR = ROOT / "table_check"
if str(SECRETS_DIR) not in sys.path:
    sys.path.append(str(SECRETS_DIR))

from connection_secrets import LAKE_USER, LAKE_PASSWORD

In [ ]:
def get_impala_connection():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={
            "user_name": LAKE_USER,
            "password": LAKE_PASSWORD,
        },
    )
    imp._init_connection()
    return imp

In [ ]:
# Инициализация Impala-соединения
imp = get_impala_connection()
print("Impala connection initialized")

In [ ]:
month_start = "2026-02-01"  # YYYY-MM-01

In [ ]:
with imp:
    sql_cnt = imp.fetch(f"""
        with params as (
            select cast('{month_start}' as date) as month_start
        ),
        feb_clients as (
            select distinct
                c.c_inn as inn,
                a.abs_agr_id as agr_id
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            join ods_alpha.scd1_trx_acq ta
              on ta.n_agr = a.n_agr
            join ods_alpha.scd1_trx t
              on t.n_trx = ta.n_trx
            join params p on 1=1
            where a.acq_class = 'SA'
              and a.abs_agr_id is not null
              and t.c_trx_class = 'SA'
              and t.c_trx_type = 'S01'
              and t.c_nter is not null
              and t.ods_deleted_flg <> '1'
              and t.cf_trx_stat <> 'R'
              and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = p.month_start
        )
        select
            cast('{month_start}' as date) as month_start,
            count(distinct agr_id) as sql_agr_cnt,
            count(distinct inn) as sql_inn_cnt
        from feb_clients
    """)
sql_cnt